# Calcium Imaging Analysis — Simplified Whole-Cell Version
**No DAPI. No soma/process split. Just whole-cell ROIs and dF/F₀ traces.**

| Cell | What it does | Why you stop here |
|------|-------------|-------------------|
| 1 | Imports + config | Set all your parameters once |
| 2 | Load one test video + show raw frames | Confirm channel loaded correctly |
| 3 | Threshold preview on single frame | Tune threshold visually |
| 4 | Segmentation preview on single frame | Check ROI detection before tracking |
| 5 | Full single video test run + overlay preview | Confirm everything works on one video |
| 6 | dF/F₀ trace preview for test video | Check traces look biological |
| 7 | Run all videos in one replicate | Spot-check before doing all replicates |
| 8 | Run all replicates + save all Excel files | Full analysis |
| 9 | Master summary + plots | Cross-replicate comparison |

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1 — Imports + Config
# Only edit this cell. Everything below is automatic.
# ═══════════════════════════════════════════════════════════════

import os, re, sys, warnings, traceback
import numpy as np
import pandas as pd
import tifffile
import cv2
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from pathlib import Path
from scipy import ndimage as ndi
from scipy.spatial.distance import cdist
from skimage import filters, measure, morphology, exposure, segmentation
from skimage.feature import peak_local_max
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter
warnings.filterwarnings('ignore')
%matplotlib inline

# ── ✏️  EDIT THESE ──────────────────────────────────────────────
MASTER_FOLDER = Path('path name')

# Pick one video to use as a test (Cells 2-6).
# Set to None to auto-pick the first file found.
TEST_VIDEO = Path ('path name')

# Pick one replicate folder name for Cell 7 spot-check
TEST_REPLICATE = 'N1'

CFG = dict(
    # ── Thresholding ─────────────────────────────────────────────
    # HOW IT WORKS — LOCAL CONTRAST NORMALISATION
    # Instead of using a raw intensity threshold, each frame is divided
    # by a heavily blurred version of itself (the 'local background').
    # This turns the question from 'is this pixel bright?' into
    # 'is this pixel brighter than its local neighbourhood?'
    # Result: dim neurons are detected just as reliably as bright ones,
    # because the ratio is the same regardless of absolute intensity.
    #
    # The only parameter you need to tune is lcn_ratio_threshold:
    #   1.10 = pixel must be 10% brighter than local background  (very sensitive)
    #   1.20 = pixel must be 20% brighter than local background  (default, good start)
    #   1.35 = pixel must be 35% brighter than local background  (strict)
    #   1.50 = pixel must be 50% brighter than local background  (very strict)
    # If too much background is captured → raise lcn_ratio_threshold
    # If dim neurons are still missed    → lower lcn_ratio_threshold toward 1.10
    #
    lcn_blur_sigma        = 50,    # Gaussian blur radius (px) for background estimate
                                   # should be larger than a whole cell (~50–100 px)
                                   # larger = smoother background, better for sparse cells
    lcn_ratio_threshold   = 1.005,  # ← MAIN TUNING KNOB
                                   # ratio = pixel / local_background
                                   # 1.05 = pixel only needs to be 5% above local bg
                                   # good starting point for dim neuron bodies
                                   # raise to 1.10–1.20 if too much background noise
                                   # lower toward 1.02 if still missing cells
    # ── Cell size filters ────────────────────────────────────────
    min_cell_area_um2   = 20.0,     # µm² — smaller objects discarded as debris
                                    # ↑ lowered from 50 — the dim-frame blob fragments
                                    #   after watershed can be small; raise if debris appears
    max_cell_area_um2   = 500000.0, # µm² — larger objects discarded as over-merged
                                    # ↑ raised dramatically — at 0.11 µm/px a whole-cell
                                    #   connected region can be very large in µm²
    min_cell_area_px    = 20,       # fallback if no pixel calibration

    # ── Watershed (splits touching cells) ────────────────────────
    watershed_footprint = 15,       # px — larger = fewer splits; smaller = more splits

    # ── Tracking ─────────────────────────────────────────────────
    max_tracking_dist_px    = 40,
    iou_threshold           = 0.20,
    gap_tolerance_frames    = 25,
    min_track_length        = 3,

    # ── Baseline / dF/F₀ ─────────────────────────────────────────
    baseline_percentile = 8,        # 8th percentile = robust minimum baseline

    # ── Analysis windows ─────────────────────────────────────────
    first_half_frames   = 31,       # frames considered "first 5 min"
    frames_per_second   = 0.1,      # 1 frame per 10 s

    # ── Output ───────────────────────────────────────────────────
    output_fps          = 5,
    overlay_alpha       = 0.50,
    conditions          = ['ctrl', 'r403c'],
    video_ext           = ['.tif', '.tiff', '.nd2'],
    fallback_um_per_px  = None,     # set e.g. 0.065 if metadata missing
)
# ── END EDIT ─────────────────────────────────────────────────────

print('✅  Config loaded.')
print(f'   Master folder : {MASTER_FOLDER}')
print(f'   Exists        : {MASTER_FOLDER.exists()}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# SHARED FUNCTIONS — run once, used by all cells below
# ═══════════════════════════════════════════════════════════════

# ── Install nd2 if needed ────────────────────────────────────────
import importlib, subprocess
if importlib.util.find_spec('nd2') is None:
    print('Installing nd2 reader...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'nd2', '-q'])
import nd2


# ── File loader (ND2 + TIFF) ─────────────────────────────────────
def _parse_to_timeseries(data):
    """
    Given a squeezed ndarray, return the calcium channel as (T, H, W) float32.
    Always uses the FIRST channel as calcium.
    """
    ndim, shape = data.ndim, data.shape
    if ndim == 2:
        return data[np.newaxis].astype(np.float32)   # single frame
    elif ndim == 3:
        if shape[0] <= 4:   # likely (C, H, W)
            return data[0][np.newaxis].astype(np.float32)
        return data.astype(np.float32)               # (T, H, W)
    elif ndim == 4:
        # Find channel axis
        c_ax = next((i for i, s in enumerate(shape[:2]) if s <= 4), 0)
        data = np.moveaxis(data, c_ax, 1)            # → (T, C, H, W)
        return data[:, 0].astype(np.float32)         # first channel
    elif ndim == 5:
        return data[:, 0, 0].astype(np.float32)      # drop Z + take ch0
    raise ValueError(f'Unexpected shape: {shape}')


def load_calcium_channel(path: Path):
    """
    Load calcium channel from ND2 or TIFF.
    Returns (frames: T×H×W float32, um_per_px: float|None)
    """
    suffix = path.suffix.lower()
    if suffix == '.nd2':
        with nd2.ND2File(str(path)) as f:
            arr = f.asarray()
            um_per_px = None
            try:
                um_per_px = float(f.voxel_size().x)
            except Exception:
                pass
            if um_per_px is None:
                try:
                    um_per_px = float(f.metadata.channels[0].volume.axesCalibration[0])
                except Exception:
                    pass
        data = np.squeeze(np.array(arr, dtype=np.float32))
        print(f'  ND2 shape: {data.shape}  µm/px={um_per_px}')
        return _parse_to_timeseries(data), um_per_px
    else:
        um_per_px = None
        with tifffile.TiffFile(str(path)) as tif:
            data = tif.asarray().astype(np.float32)
            try:
                if tif.ome_metadata:
                    import xml.etree.ElementTree as ET
                    root = ET.fromstring(tif.ome_metadata)
                    ns = {'ome': 'http://www.openmicroscopy.org/Schemas/OME/2016-06'}
                    px = root.find('.//ome:Pixels', ns)
                    if px is not None:
                        x = px.get('PhysicalSizeX')
                        if x: um_per_px = float(x)
            except Exception: pass
            if um_per_px is None:
                try:
                    page = tif.pages[0]
                    xres = page.tags.get('XResolution')
                    unit = page.tags.get('ResolutionUnit')
                    if xres:
                        val = xres.value
                        if isinstance(val, tuple):
                            val = val[0] / val[1] if val[1] != 0 else val[0]
                        if val > 0:
                            uv = unit.value if unit else 2
                            if uv == 3: um_per_px = 1e4 / float(val)
                            elif uv == 2: um_per_px = 25400.0 / float(val)
                except Exception: pass
        data = np.squeeze(data)
        print(f'  TIFF shape: {data.shape}  µm/px={um_per_px}')
        return _parse_to_timeseries(data), um_per_px


# ── Threshold ────────────────────────────────────────────────────

def compute_threshold(ref_frame, mode, manual_val):
    """
    mode: 'manual'  → use manual_val directly
          'auto'    → Otsu on foreground pixels, manual_val as floor
          a float   → 85th percentile (or similar) of frame,
                      but also computes a median-scaled estimate and
                      takes the MAX of the two — so bright videos
                      automatically get a stricter threshold.
                      manual_val still acts as absolute floor.
    """
    if mode == 'manual':
        return float(manual_val)

    if mode == 'auto':
        bg_cut = float(np.percentile(ref_frame, 50))
        fg = ref_frame[ref_frame > bg_cut]
        if len(fg) == 0:
            return float(manual_val)
        r8 = exposure.rescale_intensity(fg, out_range=(0, 255)).astype(np.uint8)
        otsu = filters.threshold_otsu(r8)
        fmin, fmax = fg.min(), fg.max()
        thresh = float(fmin + (otsu / 255.0) * (fmax - fmin))
        return max(thresh, float(manual_val))

    # Percentile + median-scaled mode
    if isinstance(mode, (int, float)):
        # Method 1: direct percentile of the frame
        percentile_thresh = float(np.percentile(ref_frame, float(mode) * 100))

        # Method 2: median-scaled estimate
        # Takes the median of foreground pixels and scales up slightly.
        # On bright frames this will naturally be higher than the percentile,
        # acting as a brightness-aware upper bound.
        bg_cut = float(np.percentile(ref_frame, 50))
        fg = ref_frame[ref_frame > bg_cut]
        if len(fg) > 0:
            median_thresh = float(np.median(fg)) * 1.05
        else:
            median_thresh = percentile_thresh

        # Take the stricter (higher) of the two
        thresh = max(percentile_thresh, median_thresh)

        return max(thresh, float(manual_val))

    return float(manual_val)

# ── Segmentation ─────────────────────────────────────────────────
def _lcn_ratio_map(frame, blur_sigma):
    """
    Local Contrast Normalisation: returns pixel / local_background.
    local_background = Gaussian-blurred version of the frame.
    Pixels where local_background < 1 are clipped to 1 to avoid div/0.
    """
    from scipy.ndimage import gaussian_filter
    local_bg = gaussian_filter(frame.astype(np.float32), sigma=blur_sigma)
    local_bg = np.clip(local_bg, 1.0, None)
    return frame.astype(np.float32) / local_bg


def segment_frame(frame, threshold, min_cell_px, max_cell_px, watershed_fp,
                  lcn_blur_sigma=50, lcn_ratio_threshold=1.20,
                  # legacy params kept for back-compat, no longer used:
                  adaptive_thresh_mode=None, adaptive_manual_floor=None,
                  is_dim_frame=False):
    """
    Local Contrast Normalisation segmentation.

    Each pixel is divided by its local background (Gaussian blur).
    Pixels with ratio > lcn_ratio_threshold are foreground.
    This makes dim and bright neurons equally detectable —
    the ratio is the same regardless of absolute frame brightness.

    lcn_blur_sigma      : blur radius; should exceed cell diameter (~50 px)
    lcn_ratio_threshold : main sensitivity knob (1.10 sensitive → 1.50 strict)
    """
    ratio_map = _lcn_ratio_map(frame, lcn_blur_sigma)
    binary    = ratio_map > lcn_ratio_threshold
    binary = morphology.remove_small_objects(binary, min_size=min_cell_px)
    binary = ndi.binary_fill_holes(binary)
    binary = morphology.binary_dilation(binary, morphology.disk(2))

    labeled, _ = ndi.label(binary)
    props_all = measure.regionprops(labeled, intensity_image=frame)
    keep = np.zeros_like(labeled)
    kept_props = []
    new_lbl = 1
    _rejected_small, _rejected_large = 0, 0
    for p in props_all:
        if p.area < min_cell_px:
            _rejected_small += 1
        elif p.area > max_cell_px:
            _rejected_large += 1
        else:
            keep[labeled == p.label] = new_lbl
            new_lbl += 1
            kept_props.append(p)
    if not kept_props and props_all:
        areas = sorted([p.area for p in props_all])
        print(f'  ⚠️  All {len(props_all)} ROIs rejected by area filter: '
              f'min_px={min_cell_px}  max_px={max_cell_px}')
        print(f'     Blob areas (px): smallest={areas[0]}  '
              f'median={areas[len(areas)//2]}  largest={areas[-1]}')
        print(f'     → {_rejected_small} too small, {_rejected_large} too large')
    return keep, kept_props

# ── ROI Tracker ──────────────────────────────────────────────────
class ROITracker:
    def __init__(self, max_dist, iou_thresh, gap_tol):
        self.max_dist = max_dist
        self.iou_thresh = iou_thresh
        self.gap_tol = gap_tol
        self.next_id = 1
        self.tracks = {}

    def update(self, props, labeled):
        if not props:
            for t in self.tracks: self.tracks[t]['absent'] += 1
            self.tracks = {tid: t for tid, t in self.tracks.items()
                           if t['absent'] <= self.gap_tol}
            return {}

        curr_centroids = np.array([p.centroid for p in props])
        curr_masks = [labeled == p.label for p in props]
        assignment, used = {}, set()

        if self.tracks:
            prev_ids = list(self.tracks.keys())
            prev_centroids = np.array([self.tracks[t]['centroid'] for t in prev_ids])
            dist_mat = cdist(curr_centroids, prev_centroids)
            pairs = sorted([(i, j, dist_mat[i, j])
                            for i in range(len(props))
                            for j in range(len(prev_ids))],
                           key=lambda x: x[2])
            for i, j, d in pairs:
                if i in assignment or prev_ids[j] in used: continue
                tid = prev_ids[j]
                accept = d <= self.max_dist
                if not accept:
                    pm = self.tracks[tid].get('mask')
                    if pm is not None:
                        inter = (curr_masks[i] & pm).sum()
                        union = (curr_masks[i] | pm).sum()
                        accept = (inter / max(union, 1)) >= self.iou_thresh
                if accept:
                    assignment[i] = tid
                    used.add(tid)

        new_tracks, label_map = {}, {}
        for i, p in enumerate(props):
            if i not in assignment:
                assignment[i] = self.next_id
                self.next_id += 1
            tid = assignment[i]
            new_tracks[tid] = {'centroid': p.centroid,
                               'mask': curr_masks[i], 'absent': 0}
            label_map[p.label] = tid

        for tid, t in self.tracks.items():
            if tid not in used:
                t['absent'] += 1
                if t['absent'] <= self.gap_tol:
                    new_tracks[tid] = t
        self.tracks = new_tracks
        return label_map


# ── dF/F₀ ────────────────────────────────────────────────────────
def compute_dff(trace, percentile):
    valid = trace[~np.isnan(trace)]
    if len(valid) == 0:
        nan = np.full_like(trace, np.nan)
        return nan, nan, np.nan, np.nan
    f0_min = float(np.nanmin(trace))
    f0_pct = float(np.nanpercentile(trace, percentile))
    def dff(f0):
        if f0 <= 0: return np.full_like(trace, np.nan)
        return np.where(~np.isnan(trace), (trace - f0) / f0, np.nan)
    return dff(f0_min), dff(f0_pct), f0_min, f0_pct


# ── Overlay frame ────────────────────────────────────────────────
CELL_CMAP = cm.get_cmap('tab20', 20)

def make_overlay_frame(ca_frame, labeled, label_to_tid, alpha):
    ca_8 = exposure.rescale_intensity(ca_frame, out_range=(0, 255)).astype(np.uint8)
    bgr  = np.stack([ca_8, ca_8, ca_8], axis=-1)
    overlay = bgr.copy()
    for frame_lbl, tid in label_to_tid.items():
        mask = labeled == frame_lbl
        c = CELL_CMAP(tid % 20)
        overlay[mask] = (int(c[2]*255), int(c[1]*255), int(c[0]*255))
    blended = cv2.addWeighted(overlay, alpha, bgr, 1 - alpha, 0)
    props = measure.regionprops(labeled)
    for p in props:
        tid = label_to_tid.get(p.label, p.label)
        cy, cx = p.centroid
        cv2.putText(blended, f'C{tid}', (int(cx), int(cy)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1, cv2.LINE_AA)
    return blended


# ── Helpers ──────────────────────────────────────────────────────
def detect_condition(filename, conditions):
    fname = filename.lower()
    for c in conditions:
        if c.lower() in fname: return c
    return 'unknown'

def detect_replicate(folder_name):
    m = re.search(r'[Nn]\d+', folder_name)
    return m.group(0).upper() if m else folder_name

def px_limits(cfg, um_per_px):
    if um_per_px and um_per_px > 0:
        px_per_um2 = 1.0 / (um_per_px ** 2)
        return (max(5, int(cfg['min_cell_area_um2'] * px_per_um2)),
                int(cfg['max_cell_area_um2'] * px_per_um2))
    return cfg['min_cell_area_px'], 999999


# ── Excel helpers ────────────────────────────────────────────────
HDR_FILL = PatternFill('solid', fgColor='1F3864')
HDR_FONT = Font(color='FFFFFF', bold=True, size=10)
ALT_FILL = PatternFill('solid', fgColor='DCE6F1')
THIN     = Side(style='thin', color='AAAAAA')
BORDER   = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)

def write_df_to_sheet(ws, df, color='1F3864'):
    fill = PatternFill('solid', fgColor=color)
    for c_idx, col in enumerate(df.columns, 1):
        cell = ws.cell(row=1, column=c_idx, value=col)
        cell.fill, cell.font = fill, HDR_FONT
        cell.alignment = Alignment(horizontal='center', wrap_text=True)
        cell.border = BORDER
    for r_idx, row in enumerate(df.itertuples(index=False), 2):
        for c_idx, val in enumerate(row, 1):
            cell = ws.cell(row=r_idx, column=c_idx,
                           value=None if (isinstance(val, float)
                                          and np.isnan(val)) else val)
            cell.fill  = ALT_FILL if r_idx % 2 == 0 else PatternFill()
            cell.border = BORDER
    for c_idx, col in enumerate(df.columns, 1):
        vals = [str(col)] + [str(v) for v in df.iloc[:, c_idx-1] if v is not None]
        ws.column_dimensions[get_column_letter(c_idx)].width = min(
            max((len(v) for v in vals), default=10) + 2, 35)
    ws.freeze_panes = 'A2'


print('✅  Shared functions loaded.')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — Load one test video + show raw frames
# STOP HERE: confirm the calcium channel loaded correctly.
# ═══════════════════════════════════════════════════════════════

# Auto-pick first file if TEST_VIDEO not set
if TEST_VIDEO is None:
    rep_dirs = sorted([d for d in MASTER_FOLDER.iterdir()
                       if d.is_dir() and re.match(r'[Nn]\d+', d.name)])
    search_dirs = rep_dirs if rep_dirs else [MASTER_FOLDER]
    for d in search_dirs:
        found = sorted([f for f in d.iterdir()
                        if f.suffix.lower() in CFG['video_ext']])
        if found:
            TEST_VIDEO = found[0]
            break

if TEST_VIDEO is None:
    raise FileNotFoundError('No video files found. Check MASTER_FOLDER in Cell 1.')

print(f'Test video: {TEST_VIDEO}')
frames_test, um_per_px_test = load_calcium_channel(TEST_VIDEO)
um_per_px_test = um_per_px_test or CFG['fallback_um_per_px']
T_test = len(frames_test)
print(f'Frames: {T_test}  (~{T_test / CFG["frames_per_second"] / 60:.1f} min)')
print(f'µm/px : {um_per_px_test}')
print(f'Frame shape: {frames_test.shape[1:]}')

# Show 4 evenly-spaced frames
sample_idx = np.linspace(0, T_test - 1, 4, dtype=int)
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, t in zip(axes, sample_idx):
    ax.imshow(frames_test[t], cmap='hot')
    ax.set_title(f'Ca²⁺  t={t}', fontsize=9)
    ax.axis('off')
plt.suptitle('Raw calcium frames — do these look correct?',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()
print('\n✅  Cell 2 done → proceed to Cell 3')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3 — LCN ratio map preview
# This shows you what the segmentation actually 'sees'.
# The ratio map should show cells as bright regions on a flat
# grey background regardless of how dim the raw image is.
# Adjust lcn_ratio_threshold and lcn_blur_sigma in Cell 1.
# ═══════════════════════════════════════════════════════════════

PREVIEW_FRAME = 0   # ← change to any frame index

ca_frame_test  = frames_test[PREVIEW_FRAME]
ratio_map_test = _lcn_ratio_map(ca_frame_test, CFG['lcn_blur_sigma'])
binary_test    = ratio_map_test > CFG['lcn_ratio_threshold']
min_px_test, max_px_test = px_limits(CFG, um_per_px_test)

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

axes[0].imshow(ca_frame_test, cmap='hot')
axes[0].set_title(f'Raw Ca²⁺  frame {PREVIEW_FRAME}', fontsize=9)
axes[0].axis('off')

# Show ratio map with a fixed scale so dim neuron bodies are visible
# vmax=1.5 means anything at 1.5× background or brighter is saturated white
axes[1].imshow(ratio_map_test, cmap='hot', vmin=0.9, vmax=1.5)
axes[1].set_title('LCN ratio map  (pixel ÷ local background)\n'
                  'Neuron bodies typically appear at ratio 1.02–1.15', fontsize=8)
axes[1].axis('off')

axes[2].imshow(binary_test, cmap='Oranges')
axes[2].set_title(f'Cell mask  (ratio > {CFG["lcn_ratio_threshold"]})', fontsize=9)
axes[2].axis('off')

# Histogram of ratio values
axes[3].hist(ratio_map_test.ravel(), bins=300, color='steelblue', alpha=0.7,
             range=(0.95, 1.6))   # zoomed in to see the neuron-body peak
axes[3].axvline(CFG['lcn_ratio_threshold'], color='red', lw=2,
                label=f'Threshold {CFG["lcn_ratio_threshold"]}')
axes[3].axvspan(0.95, CFG['lcn_ratio_threshold'], alpha=0.08, color='red',
                label='below threshold (background)')
axes[3].set_xlabel('LCN ratio (pixel / local_bg)')
axes[3].set_ylabel('Pixel count')
axes[3].set_title('Ratio histogram  ZOOMED 0.95–1.6\n'
                  'Neuron bodies = small shoulder/bump between 1.02–1.15')
axes[3].set_yscale('log')
axes[3].legend(fontsize=8)
# Also print the actual ratio values at the neuron body pixels for guidance
_cell_ratios = ratio_map_test[ratio_map_test > 1.01]
if len(_cell_ratios):
    print(f'Ratio at foreground pixels (>1.01): '
          f'p10={np.percentile(_cell_ratios,10):.3f}  '
          f'p50={np.percentile(_cell_ratios,50):.3f}  '
          f'p90={np.percentile(_cell_ratios,90):.3f}  '
          f'max={_cell_ratios.max():.3f}')
    print('→ Set lcn_ratio_threshold just above the p10 value to catch dim cells')

plt.suptitle('LCN Preview — adjust lcn_ratio_threshold in Cell 1',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

# Area-limit info
if um_per_px_test:
    _min_px_disp, _max_px_disp = px_limits(CFG, um_per_px_test)
    print(f'LCN ratio threshold : {CFG["lcn_ratio_threshold"]}')
    print(f'LCN blur sigma      : {CFG["lcn_blur_sigma"]} px')
    print(f'min_cell_area_um2   : {CFG["min_cell_area_um2"]} µm²  → {_min_px_disp} px  at {um_per_px_test:.3f} µm/px')
    print(f'max_cell_area_um2   : {CFG["max_cell_area_um2"]} µm²  → {_max_px_disp} px')
    print()
    print('Tuning guide:')
    print('  • Too much background in mask → raise lcn_ratio_threshold (e.g. 1.25, 1.30)')
    print('  • Dim cells missing from mask → lower lcn_ratio_threshold (e.g. 1.15, 1.10)')
    print('  • Mask broken into fragments  → raise lcn_blur_sigma (e.g. 75, 100)')
    print('  • Mask merging separate cells → lower lcn_blur_sigma (e.g. 30, 20)')

print('\n✅  Cell 3 done → proceed to Cell 4')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4 — Segmentation preview on single frame
# STOP HERE: check ROI detection.
# Adjust min_cell_area_um2, max_cell_area_um2, watershed_footprint
# in Cell 1 then re-run.
# ═══════════════════════════════════════════════════════════════
# ═══════════════════════════════════════════════════════════════
# CELL 4 — Segmentation preview on single frame
# STOP HERE: check ROI detection.
# Adjust min_cell_area_um2, max_cell_area_um2 in Cell 1 then re-run.
# ═══════════════════════════════════════════════════════════════

PREVIEW_FRAME = 10   # ← change to inspect a different frame

min_px_test, max_px_test = px_limits(CFG, um_per_px_test)
ca_frame_test  = frames_test[PREVIEW_FRAME]
labeled_test, props_test = segment_frame(ca_frame_test, None,
                                         min_px_test, max_px_test,
                                         CFG['watershed_footprint'],
                                         lcn_blur_sigma=CFG['lcn_blur_sigma'],
                                         lcn_ratio_threshold=CFG['lcn_ratio_threshold'])

print(f'Frame {PREVIEW_FRAME}: {len(props_test)} ROIs detected')
if um_per_px_test:
    areas = [p.area * um_per_px_test**2 for p in props_test]
    if areas:
        print(f'ROI areas (µm²): min={min(areas):.0f}  '
              f'max={max(areas):.0f}  mean={np.mean(areas):.0f}')
    else:
        # Re-run with no area filter to show what sizes are actually coming out
        import copy as _copy
        _lbl0, _props0 = segment_frame(ca_frame_test, None,
                                       0, 999999999,
                                       CFG['watershed_footprint'],
                                       lcn_blur_sigma=CFG['lcn_blur_sigma'],
                                       lcn_ratio_threshold=CFG['lcn_ratio_threshold'])
        _min_px, _max_px = px_limits(CFG, um_per_px_test)
        print(f'❌  0 ROIs survived the area filter')
        print(f'   Area limits: {CFG["min_cell_area_um2"]}–{CFG["max_cell_area_um2"]} µm²'
              f'  →  {_min_px}–{_max_px} px')
        if _props0:
            _areas_px  = sorted([p.area for p in _props0])
            _areas_um2 = ([round(a * um_per_px_test**2, 1) for a in _areas_px]
                          if um_per_px_test else _areas_px)
            print(f'   Blobs WITHOUT filter: {len(_props0)} found')
            print(f'   Their areas (µm²): {_areas_um2[:8]} ...'
                  f'  largest={_areas_um2[-1]}')
            print('   → Adjust min_cell_area_um2 / max_cell_area_um2 in Cell 1 to match')
        else:
            print('   No blobs at all — threshold is too HIGH for this frame')
            print(f'   Threshold was {thresh_test:.1f}; try lowering manual_threshold')

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Raw
axes[0].imshow(ca_frame_test, cmap='hot')
axes[0].set_title('Raw Ca²⁺', fontsize=10)
axes[0].axis('off')

# Outlines
ca_8 = exposure.rescale_intensity(ca_frame_test, out_range=(0,255)).astype(np.uint8)
rgb  = np.stack([ca_8, ca_8, ca_8], axis=-1)
for p in props_test:
    for cnt in measure.find_contours(labeled_test == p.label, 0.5):
        rr = np.clip(cnt[:,0].astype(int), 0, rgb.shape[0]-1)
        cc = np.clip(cnt[:,1].astype(int), 0, rgb.shape[1]-1)
        rgb[rr, cc] = [255, 140, 0]
    cy, cx = p.centroid
    axes[1].text(cx, cy, str(p.label), fontsize=7,
                 color='white', ha='center', va='center')
axes[1].imshow(rgb)
axes[1].set_title(f'{len(props_test)} ROIs detected (orange outlines)', fontsize=10)
axes[1].axis('off')

# Colour map
label_rgb = np.zeros((*labeled_test.shape, 3), dtype=np.uint8)
for p in props_test:
    mask = labeled_test == p.label
    c = CELL_CMAP(p.label % 20)
    label_rgb[mask] = [int(c[0]*255), int(c[1]*255), int(c[2]*255)]
axes[2].imshow(label_rgb)
axes[2].set_title('ROI colour map', fontsize=10)
axes[2].axis('off')

plt.suptitle(f'Segmentation Preview — Frame {PREVIEW_FRAME}',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print('\n✅  Cell 4 done → proceed to Cell 5')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 5 — Full single video test run + overlay preview
# STOP HERE: confirm full pipeline works on one video.
# ═══════════════════════════════════════════════════════════════

out_dir_test = MASTER_FOLDER / 'Results' / 'test_preview'
out_dir_test.mkdir(parents=True, exist_ok=True)


def analyse_video(video_path, cfg, out_dir, fallback_um_per_px=None):
    """
    Full analysis of one video.
    Returns (df_frames, df_trace_raw, df_trace_dff_min,
             df_trace_dff_pct, df_peaks, video_row)
    """
    frames, um_per_px = load_calcium_channel(video_path)
    if um_per_px is None: um_per_px = fallback_um_per_px
    T = len(frames)
    print(f'  Frames: {T}  µm/px={um_per_px}')

    min_px, max_px = px_limits(cfg, um_per_px)
    print(f'  Cell area: {min_px}–{max_px} px')

    tracker    = ROITracker(cfg['max_tracking_dist_px'],
                            cfg['iou_threshold'],
                            cfg['gap_tolerance_frames'])
    raw_store  = {}
    area_store = {}
    obj_rows   = []
    overlay_frames = []

    print(f'  LCN ratio threshold : {cfg["lcn_ratio_threshold"]}')
    print(f'  LCN blur sigma      : {cfg["lcn_blur_sigma"]} px')
    frame_medians = np.array([float(np.median(f)) for f in frames])
    dim_cutoff    = float(np.percentile(frame_medians, 25))   # kept for Excel reporting

    for t, ca_frame in enumerate(frames):
        is_dim = frame_medians[t] < dim_cutoff   # for Excel stats only
        labeled, props = segment_frame(ca_frame, None, min_px, max_px,
                               cfg['watershed_footprint'],
                               lcn_blur_sigma=cfg['lcn_blur_sigma'],
                               lcn_ratio_threshold=cfg['lcn_ratio_threshold'])
        label_map  = tracker.update(props, labeled)
        mean_field = float(ca_frame.mean())

        for p in props:
            tid = label_map.get(p.label)
            if tid is None: continue
            if tid not in raw_store:
                raw_store[tid]  = np.full(T, np.nan, dtype=np.float32)
                area_store[tid] = np.full(T, np.nan, dtype=np.float32)
            raw_store[tid][t]  = p.mean_intensity
            area_store[tid][t] = p.area
            area_um2 = p.area * (um_per_px**2) if um_per_px else np.nan
            obj_rows.append({
                'frame'             : t,
                'time_min'          : t / cfg['frames_per_second'] / 60.0,
                'track_id'          : tid,
                'raw_intensity_647' : p.mean_intensity,
                'mean_field_intensity': mean_field,
                'area_px2'          : p.area,
                'area_um2'          : area_um2,
                'centroid_y'        : p.centroid[0],
                'centroid_x'        : p.centroid[1],
                'is_dim_frame'      : is_dim,
            })
        overlay_frames.append(
            make_overlay_frame(ca_frame, labeled, label_map,
                               cfg['overlay_alpha']))

    # Filter short tracks
    raw_store = {tid: arr for tid, arr in raw_store.items()
                 if int(np.sum(~np.isnan(arr))) >= cfg['min_track_length']}

    # dF/F₀
    dff_min_store, dff_pct_store = {}, {}
    f0_min_store,  f0_pct_store  = {}, {}
    for tid, arr in raw_store.items():
        dm, dp, fm, fp = compute_dff(arr, cfg['baseline_percentile'])
        dff_min_store[tid], dff_pct_store[tid] = dm, dp
        f0_min_store[tid],  f0_pct_store[tid]  = fm, fp

    # Per-frame DataFrame
    df_frames = pd.DataFrame(obj_rows)
    if len(df_frames):
        df_frames['video'] = video_path.name
        dff_min_rows, dff_pct_rows, f0min_rows, f0pct_rows = [], [], [], []
        for _, row in df_frames.iterrows():
            tid, t = row['track_id'], int(row['frame'])
            dff_min_rows.append(dff_min_store[tid][t] if tid in dff_min_store else np.nan)
            dff_pct_rows.append(dff_pct_store[tid][t] if tid in dff_pct_store else np.nan)
            f0min_rows.append(f0_min_store.get(tid, np.nan))
            f0pct_rows.append(f0_pct_store.get(tid, np.nan))
        df_frames['f0_global_min']   = f0min_rows
        df_frames['f0_percentile']   = f0pct_rows
        df_frames['dff0_global_min'] = dff_min_rows
        df_frames['dff0_percentile'] = dff_pct_rows

    # Trace DataFrames
    def make_trace_df(store):
        d = {f'C{tid}': arr for tid, arr in sorted(store.items())}
        df = pd.DataFrame(d)
        df.insert(0, 'time_min',
                  np.arange(T) / cfg['frames_per_second'] / 60.0)
        df.insert(0, 'frame', np.arange(T))
        return df

    df_trace_raw     = make_trace_df(raw_store)
    df_trace_dff_min = make_trace_df(dff_min_store)
    df_trace_dff_pct = make_trace_df(dff_pct_store)

    # Peak per ROI
    peak_rows = []
    for tid in sorted(raw_store):
        raw, dm, dp = raw_store[tid], dff_min_store[tid], dff_pct_store[tid]
        area = area_store.get(tid, np.full(T, np.nan))
        n_det = int(np.sum(~np.isnan(raw)))
        area_um2_mean = float(np.nanmean(area)) * (um_per_px**2) \
                        if um_per_px else np.nan
        h = cfg['first_half_frames']
        peak_rows.append({
            'track_id'                  : tid,
            'f0_global_min'             : f0_min_store.get(tid, np.nan),
            'f0_percentile'             : f0_pct_store.get(tid, np.nan),
            'peak_dff0_global_min'      : float(np.nanmax(dm)),
            'mean_dff0_global_min'      : float(np.nanmean(dm)),
            'peak_dff0_percentile'      : float(np.nanmax(dp)),
            'mean_dff0_percentile'      : float(np.nanmean(dp)),
            'mean_raw_intensity'        : float(np.nanmean(raw)),
            'mean_raw_intensity_first31': float(np.nanmean(raw[:h])),
            'mean_area_px2'             : float(np.nanmean(area)),
            'mean_area_um2'             : area_um2_mean,
            'n_frames_detected'         : n_det,
            'pct_frames_detected'       : round(100 * n_det / T, 1),
            'video'                     : video_path.name,
        })
    df_peaks = pd.DataFrame(peak_rows)

    df_frames_h = df_frames[df_frames['frame'] < cfg['first_half_frames']] \
                  if len(df_frames) else df_frames
    video_row = {
        'video'                          : video_path.name,
        'total_frames'                   : T,
        'duration_min'                   : round(T / cfg['frames_per_second'] / 60, 2),
        'um_per_px'                      : um_per_px,
        'lcn_ratio_threshold'            : cfg['lcn_ratio_threshold'],
        'n_rois'                         : len(raw_store),
        'n_dim_frames'                   : int(np.sum(frame_medians < dim_cutoff))
                                           if frame_medians is not None else 0,
        'n_bright_frames'                : int(np.sum(frame_medians >= dim_cutoff))
                                           if frame_medians is not None else T,
        'mean_raw_intensity_full'        : df_frames['raw_intensity_647'].mean()
                                           if len(df_frames) else np.nan,
        'mean_field_intensity_full'      : df_frames['mean_field_intensity'].mean()
                                           if len(df_frames) else np.nan,
        'mean_area_um2_full'             : df_frames['area_um2'].mean()
                                           if len(df_frames) else np.nan,
        'mean_peak_dff0_min_full'        : df_peaks['peak_dff0_global_min'].mean()
                                           if len(df_peaks) else np.nan,
        'mean_peak_dff0_pct_full'        : df_peaks['peak_dff0_percentile'].mean()
                                           if len(df_peaks) else np.nan,
        'mean_raw_intensity_first31'     : df_frames_h['raw_intensity_647'].mean()
                                           if len(df_frames_h) else np.nan,
    }

    # Save overlay video
    if overlay_frames:
        out_vid = out_dir / (video_path.stem + '_overlay.mp4')
        h2, w2 = overlay_frames[0].shape[:2]
        writer = cv2.VideoWriter(str(out_vid),
                                 cv2.VideoWriter_fourcc(*'mp4v'),
                                 cfg['output_fps'], (w2, h2))
        for f in overlay_frames: writer.write(f)
        writer.release()
        print(f'  Overlay → {out_vid.name}')

    return df_frames, df_trace_raw, df_trace_dff_min, df_trace_dff_pct, \
           df_peaks, video_row


print(f'Running: {TEST_VIDEO.name} ...')
(df_frames_test, df_trace_raw_test, df_trace_dmin_test,
 df_trace_dpct_test, df_peaks_test, vid_row_test) = analyse_video(
    TEST_VIDEO, CFG, out_dir_test, CFG['fallback_um_per_px'])

print(f'\nROIs tracked: {vid_row_test["n_rois"]}')
for k, v in vid_row_test.items(): print(f'  {k:40s} {v}')

# Show first overlay frame
overlay_path = out_dir_test / (TEST_VIDEO.stem + '_overlay.mp4')
cap = cv2.VideoCapture(str(overlay_path))
ret, frame_bgr = cap.read()
cap.release()
if ret:
    fig, ax = plt.subplots(figsize=(9, 7))
    ax.imshow(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
    ax.set_title('Overlay — first frame (each colour = one tracked cell)',
                 fontsize=10)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

print('\n✅  Cell 5 done → proceed to Cell 6')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 6 — dF/F₀ trace preview
# STOP HERE: check traces look biological before running all videos.
# ═══════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(18, max(4, len(df_trace_dmin_test.columns) * 0.5)))

for ax, (df_tr, title) in zip(axes, [
    (df_trace_dmin_test, 'dF/F₀  (global min baseline)'),
    (df_trace_dpct_test, 'dF/F₀  (8th percentile baseline)'),
]):
    offset  = 0
    spacing = 1.5
    t_axis  = df_tr['time_min'].values
    cell_cols = [c for c in df_tr.columns if c not in ('frame', 'time_min')]
    for idx, col in enumerate(cell_cols):
        vals  = df_tr[col].values.astype(float)
        valid = ~np.isnan(vals)
        color = CELL_CMAP(idx % 20)
        if valid.any():
            ax.plot(t_axis[valid], vals[valid] + offset,
                    color=color, lw=0.9, alpha=0.9)
            ax.text(0, offset + 0.05, col, fontsize=6.5, color=color)
        offset += spacing
    ax.set_xlabel('Time (min)', fontsize=10)
    ax.set_ylabel('dF/F₀ (offset per cell)', fontsize=10)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False)

fig.suptitle(f'dF/F₀ Traces — {TEST_VIDEO.name}',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

n_cells = len([c for c in df_trace_dmin_test.columns
               if c not in ('frame', 'time_min')])
print(f'Cells with traces: {n_cells}')
print('\n✅  Cell 6 done → proceed to Cell 7')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 7 — Run all videos in one replicate (spot-check)
# STOP HERE: review before running all replicates.
# ═══════════════════════════════════════════════════════════════

results_root = MASTER_FOLDER / 'Results'
results_root.mkdir(exist_ok=True)


def run_replicate(rep_folder, cfg, results_root):
    replicate = detect_replicate(rep_folder.name)
    rep_out   = results_root / replicate
    rep_out.mkdir(parents=True, exist_ok=True)

    vid_files = sorted([f for f in rep_folder.iterdir()
                        if f.suffix.lower() in cfg['video_ext']])
    if not vid_files:
        print(f'  No videos in {rep_folder.name}')
        return None

    rep_frames, rep_trace_raw  = [], []
    rep_trace_dmin, rep_trace_dpct = [], []
    rep_peaks, rep_vid_rows    = [], []

    for vid_path in vid_files:
        condition = detect_condition(vid_path.name, cfg['conditions'])
        print(f'  [{replicate}] {vid_path.name}  ({condition})')
        try:
            df_fr, df_tr_raw, df_tr_dmin, df_tr_dpct, df_pk, vid_row = \
                analyse_video(vid_path, cfg, rep_out,
                              cfg.get('fallback_um_per_px'))
            for df in [df_fr, df_pk]:
                if len(df):
                    df['condition'] = condition
                    df['replicate'] = replicate
            # Prefix trace columns with video stem to avoid clashes
            rename = {c: f'{vid_path.stem}__{c}'
                      for c in df_tr_raw.columns
                      if c not in ('frame', 'time_min')}
            for df in [df_tr_raw, df_tr_dmin, df_tr_dpct]:
                df.rename(columns=rename, inplace=True)
            vid_row['condition'] = condition
            vid_row['replicate'] = replicate
            rep_frames.append(df_fr)
            rep_trace_raw.append(df_tr_raw)
            rep_trace_dmin.append(df_tr_dmin)
            rep_trace_dpct.append(df_tr_dpct)
            rep_peaks.append(df_pk)
            rep_vid_rows.append(vid_row)
        except Exception as e:
            print(f'  ⚠️  ERROR {vid_path.name}: {e}')
            traceback.print_exc()

    if not rep_vid_rows: return None

    def merge_traces(trace_list):
        if not trace_list: return pd.DataFrame()
        base = trace_list[0][['frame', 'time_min']].copy()
        for df in trace_list:
            for col in df.columns:
                if col not in ('frame', 'time_min') and col not in base.columns:
                    base[col] = np.nan
                if col not in ('frame', 'time_min'):
                    mask = base['frame'].isin(df['frame'])
                    base.loc[mask, col] = df.set_index('frame').reindex(
                        base.loc[mask, 'frame'])[col].values
        return base

    all_fr  = pd.concat(rep_frames, ignore_index=True) if rep_frames  else pd.DataFrame()
    all_pk  = pd.concat(rep_peaks,  ignore_index=True) if rep_peaks   else pd.DataFrame()
    tr_raw  = merge_traces(rep_trace_raw)
    tr_dmin = merge_traces(rep_trace_dmin)
    tr_dpct = merge_traces(rep_trace_dpct)

    # Save Excel
    wb = Workbook()
    wb.remove(wb.active)
    def add(name, df, color='1F3864'):
        ws = wb.create_sheet(name)
        if len(df): write_df_to_sheet(ws, df, color)

    if len(all_fr):   add('Per_Frame_Stats',     all_fr,  '1F3864')
    add('Full_Video_Summary', pd.DataFrame(rep_vid_rows), '203864')
    if len(all_fr):
        h = cfg['first_half_frames']
        df_first = (all_fr[all_fr['frame'] < h]
                    .groupby(['video', 'track_id'], as_index=False)
                    .agg(mean_raw_intensity   = ('raw_intensity_647', 'mean'),
                         mean_dff0_global_min = ('dff0_global_min',   'mean'),
                         mean_dff0_percentile = ('dff0_percentile',   'mean'),
                         mean_area_um2        = ('area_um2',          'mean')))
        add('First_5min_Summary',    df_first, '1F4E79')
    if len(tr_raw):   add('Cell_Traces_Raw',      tr_raw,  '375623')
    if len(tr_dmin):  add('Cell_Traces_dFF0_Min', tr_dmin, '375623')
    if len(tr_dpct):  add('Cell_Traces_dFF0_Pct', tr_dpct, '375623')
    if len(all_pk):   add('Peak_Per_Cell',        all_pk,  '7B2C2C')

    excel_path = rep_out / f'{replicate}_results.xlsx'
    wb.save(str(excel_path))
    print(f'  Excel → {excel_path}')
    return {'vid_rows': rep_vid_rows, 'peaks': all_pk}


# Run spot-check replicate
rep_dir = MASTER_FOLDER / TEST_REPLICATE
if not rep_dir.exists():
    rep_dirs = sorted([d for d in MASTER_FOLDER.iterdir()
                       if d.is_dir() and re.match(r'[Nn]\d+', d.name)])
    rep_dir  = rep_dirs[0] if rep_dirs else MASTER_FOLDER

print(f'Spot-check replicate: {rep_dir.name}')
spot = run_replicate(rep_dir, CFG, results_root)

if spot:
    print(f'\nVideos: {len(spot["vid_rows"])}')
    cols = ['video', 'condition', 'n_rois',
            'mean_peak_dff0_min_full', 'threshold']
    print(pd.DataFrame(spot['vid_rows'])[cols].to_string(index=False))

print('\n✅  Cell 7 done → proceed to Cell 8')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 8 — Run ALL replicates + save Excel files
# Full analysis — go get a coffee ☕
# ═══════════════════════════════════════════════════════════════

master_vid_rows = []
master_peaks    = []

rep_folders = sorted([d for d in MASTER_FOLDER.iterdir()
                      if d.is_dir()
                      and re.match(r'[Nn]\d+', d.name)
                      and d.name != 'Results'])
if not rep_folders:
    print('No N1/N2/N3 subfolders — treating master as single replicate.')
    rep_folders = [MASTER_FOLDER]

for rep_folder in rep_folders:
    print(f'\n{"="*55}')
    print(f'  Replicate: {rep_folder.name}')
    print(f'{"="*55}')
    result = run_replicate(rep_folder, CFG, results_root)
    if result is None: continue
    master_vid_rows.extend(result['vid_rows'])
    if len(result['peaks']): master_peaks.append(result['peaks'])

print(f'\nTotal videos processed: {len(master_vid_rows)}')
print('\n✅  Cell 8 done → proceed to Cell 9')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 9 — Master summary Excel + cross-replicate plots
# ═══════════════════════════════════════════════════════════════

if not master_vid_rows:
    print('No data — run Cell 8 first.')
else:
    df_master_vid   = pd.DataFrame(master_vid_rows)
    df_master_peaks = pd.concat(master_peaks, ignore_index=True) \
                      if master_peaks else pd.DataFrame()

    # Save master Excel
    wb = Workbook()
    wb.remove(wb.active)
    def add_m(name, df, color='1F3864'):
        ws = wb.create_sheet(name)
        if len(df): write_df_to_sheet(ws, df, color)

    add_m('All_Videos_Summary', df_master_vid,   '1F3864')
    add_m('All_Peaks_Per_Cell', df_master_peaks, '203864')

    if len(df_master_peaks) and 'condition' in df_master_peaks.columns:
        bio = (df_master_peaks.groupby('condition')
               .agg(
                   mean_peak_dff0_global_min = ('peak_dff0_global_min',  'mean'),
                   sem_peak_dff0_global_min  = ('peak_dff0_global_min',  'sem'),
                   mean_peak_dff0_percentile = ('peak_dff0_percentile',  'mean'),
                   sem_peak_dff0_percentile  = ('peak_dff0_percentile',  'sem'),
                   mean_raw_intensity        = ('mean_raw_intensity',    'mean'),
                   mean_area_um2             = ('mean_area_um2',         'mean'),
                   n_cells                   = ('track_id',              'count'),
                   n_videos                  = ('video',                 'nunique'),
               ).reset_index())
        add_m('Condition_Summary', bio, '4A235B')

    add_m('README', pd.DataFrame({'Note': [
        'All_Videos_Summary : one row per video',
        'All_Peaks_Per_Cell : one row per tracked whole-cell ROI',
        'Condition_Summary  : mean±SEM by condition',
        'C = whole-cell ROI (no soma/process distinction)',
        'dff0_global_min  : baseline = min intensity across full video',
        'dff0_percentile  : baseline = 8th percentile across full video',
    ]}), '555555')

    master_excel = MASTER_FOLDER / 'master_summary_simple.xlsx'
    wb.save(str(master_excel))
    print(f'Master Excel saved: {master_excel}')

    # Cross-replicate bar plots
    if len(df_master_peaks) and 'condition' in df_master_peaks.columns:
        conditions = sorted(df_master_peaks['condition'].unique())
        metrics = [
            ('peak_dff0_global_min',  'Peak dF/F₀ (global min)'),
            ('peak_dff0_percentile',  'Peak dF/F₀ (8th pct)'),
            ('mean_raw_intensity',    'Mean raw intensity'),
        ]
        fig, axes = plt.subplots(1, 3, figsize=(14, 5))
        colors = plt.cm.Set2(np.linspace(0, 1, len(conditions)))
        for ax, (metric, ylabel) in zip(axes, metrics):
            means = [df_master_peaks[df_master_peaks['condition']==c][metric].mean()
                     for c in conditions]
            sems  = [df_master_peaks[df_master_peaks['condition']==c][metric].sem()
                     for c in conditions]
            ax.bar(conditions, means, yerr=sems, capsize=5,
                   color=colors[:len(conditions)], alpha=0.85)
            ax.set_ylabel(ylabel, fontsize=9)
            ax.set_title(ylabel, fontsize=9, fontweight='bold')
            ax.spines[['top', 'right']].set_visible(False)

        plt.suptitle('Cross-replicate Summary (mean ± SEM)',
                     fontsize=12, fontweight='bold')
        plt.tight_layout()
        plot_path = results_root / 'cross_replicate_summary_simple.png'
        plt.savefig(str(plot_path), dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Plot saved: {plot_path}')

    print('\n✅  Analysis complete!')
    print(f'   Results : {results_root}')
    print(f'   Master  : {master_excel}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 10 — Different Summary
# ═══════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

RESULTS_DIR = Path('path name')

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
OUT_FILE = RESULTS_DIR.parent / f'frame_intensity_traces_{timestamp}.xlsx'

all_data = {}  # { (replicate, condition, video) : { 'cell': Series, 'field': Series } }

for rep_folder in sorted(RESULTS_DIR.iterdir()):
    if not rep_folder.is_dir():
        continue
    replicate = rep_folder.name

    for xlsx in sorted(rep_folder.rglob('*.xlsx')):
        if xlsx.name.startswith('~$'):
            print(f'  ⚠ Skipping temp file: {xlsx.name}')
            continue

        try:
            df = pd.read_excel(xlsx, sheet_name='Per_Frame_Stats')

            # Normalise condition labels in case of any capitalisation inconsistencies
            df['condition'] = df['condition'].str.strip().str.lower()
            df['condition'] = df['condition'].replace({
                'control': 'ctrl',
                'r403c'  : 'r403c',
            })

            for (condition, video), grp in df.groupby(['condition', 'video']):
                cell_avg  = grp.groupby('frame')['raw_intensity_647'].mean()
                field_avg = grp.groupby('frame')['mean_field_intensity'].mean()
                all_data[(replicate, condition, str(video))] = {
                    'cell' : cell_avg,
                    'field': field_avg
                }

            print(f'  ✓ [{replicate}] {xlsx.stem}')

        except Exception as e:
            print(f'  ✗ [{replicate}] {xlsx.stem} — {e}')

# Build time axis
n_frames = 61
time_min = pd.Series(
    np.arange(n_frames) / CFG['frames_per_second'] / 60.0,
    name='time_min'
)

replicates  = sorted(set(k[0] for k in all_data))
conditions  = sorted(set(k[1] for k in all_data))

with pd.ExcelWriter(OUT_FILE, engine='openpyxl') as writer:

    # --- Per-replicate tabs, split by condition ---
    for rep in replicates:
        for cond in conditions:
            rep_cond_data = {
                k: v for k, v in all_data.items()
                if k[0] == rep and k[1] == cond
            }
            if not rep_cond_data:
                continue  # skip if this replicate has no data for this condition

            for metric, sheet_suffix in [('cell',  'cell_avg'),
                                         ('field', 'field_intensity')]:
                df_out = pd.DataFrame({
                    vid: v[metric] for (_, _, vid), v in rep_cond_data.items()
                })
                df_out.index.name = 'frame'
                df_out.insert(0, 'time_min', time_min[:len(df_out)])
                df_out.to_excel(writer, sheet_name=f'{rep}_{cond}_{sheet_suffix}')

        print(f'  → Sheets written for {rep}')

    # --- Summary: per condition, mean ± SEM across replicates ---
    for cond in conditions:
        for metric, sheet_suffix in [('cell',  'cell_avg'),
                                     ('field', 'field_intensity')]:
            rep_means = {}
            for rep in replicates:
                rep_cond_data = {
                    k: v for k, v in all_data.items()
                    if k[0] == rep and k[1] == cond
                }
                if not rep_cond_data:
                    continue
                # Average all videos within this replicate+condition
                traces = pd.DataFrame({
                    vid: v[metric] for (_, _, vid), v in rep_cond_data.items()
                })
                rep_means[rep] = traces.mean(axis=1)

            if not rep_means:
                continue

            rep_mean_df = pd.DataFrame(rep_means)

            summary = pd.DataFrame({'time_min': time_min[:len(rep_mean_df)]})
            for rep in replicates:
                if rep in rep_mean_df.columns:
                    summary[f'mean_{rep}'] = rep_mean_df[rep].values
            summary['grand_mean'] = rep_mean_df.mean(axis=1).values
            summary['sem']        = rep_mean_df.sem(axis=1).values
            summary.index.name    = 'frame'

            summary.to_excel(writer, sheet_name=f'summary_{cond}_{sheet_suffix}')

print(f'\n✅ Saved → {OUT_FILE.name}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 11 — Build summary from existing Excel outputs for dF/F0 peak means
# ═══════════════════════════════════════════════════════════════

RESULTS_PATH = Path('path name')
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

# ── Read Peak_Per_Cell from all per-replicate Excel files ────────
peak_dfs = []
for rep_folder in ['N1', 'N2', 'N3']:
    rep_path = RESULTS_PATH / rep_folder
    if not rep_path.exists():
        print(f'  Folder not found: {rep_path}')
        continue
    for xlsx in sorted(rep_path.glob('*.xlsx')):
        try:
            df = pd.read_excel(xlsx, sheet_name='Peak_Per_Cell')
            peak_dfs.append(df)
            print(f'  Loaded: {rep_folder}/{xlsx.name}  ({len(df)} ROIs)')
        except Exception as e:
            print(f'  Skipped {xlsx.name}: {e}')

if not peak_dfs:
    print('No data found — check RESULTS_PATH and folder names.')
else:
    df_master_peaks = pd.concat(peak_dfs, ignore_index=True)
    print(f'\nTotal ROIs loaded: {len(df_master_peaks)}')
    print(df_master_peaks.groupby(['condition', 'replicate'])['track_id'].count())

    wb = Workbook()
    wb.remove(wb.active)

    def add_m(name, df, color='1F3864'):
        ws = wb.create_sheet(name)
        if len(df): write_df_to_sheet(ws, df, color)

    add_m('All_Peaks_Per_Cell', df_master_peaks, '203864')

    # ── Technical replicates: mean per video ────────────────────
    tech = (df_master_peaks
            .groupby(['condition', 'replicate', 'video'])
            .agg(
                mean_peak_dff0_global_min = ('peak_dff0_global_min', 'mean'),
                sem_peak_dff0_global_min  = ('peak_dff0_global_min', 'sem'),
                mean_peak_dff0_percentile = ('peak_dff0_percentile', 'mean'),
                sem_peak_dff0_percentile  = ('peak_dff0_percentile', 'sem'),
                mean_raw_intensity        = ('mean_raw_intensity',   'mean'),
                mean_area_um2             = ('mean_area_um2',        'mean'),
                n_cells                   = ('track_id',             'count'),
            )
            .reset_index())
    add_m('Technical_Replicates', tech, '1F4E79')

    # ── Biological replicates: videos averaged first, then per N ─
    bio_rep = (df_master_peaks
               .groupby(['condition', 'replicate', 'video'])
               .agg(
                   peak_dff0_global_min = ('peak_dff0_global_min', 'mean'),
                   peak_dff0_percentile = ('peak_dff0_percentile', 'mean'),
                   mean_raw_intensity   = ('mean_raw_intensity',   'mean'),
                   mean_area_um2        = ('mean_area_um2',        'mean'),
                   n_cells              = ('track_id',             'count'),
               )
               .reset_index()
               .groupby(['condition', 'replicate'])
               .agg(
                   mean_peak_dff0_global_min = ('peak_dff0_global_min', 'mean'),
                   sem_peak_dff0_global_min  = ('peak_dff0_global_min', 'sem'),
                   mean_peak_dff0_percentile = ('peak_dff0_percentile', 'mean'),
                   sem_peak_dff0_percentile  = ('peak_dff0_percentile', 'sem'),
                   mean_raw_intensity        = ('mean_raw_intensity',   'mean'),
                   mean_area_um2             = ('mean_area_um2',        'mean'),
                   total_cells               = ('n_cells',              'sum'),
                   n_videos                  = ('peak_dff0_global_min', 'count'),
               )
               .reset_index())
    add_m('Biological_Replicates', bio_rep, '1F5C2E')

    # ── Condition summary: N1/N2/N3 averaged equally ─────────────
    cond = (bio_rep
            .groupby('condition')
            .agg(
                mean_peak_dff0_global_min = ('mean_peak_dff0_global_min', 'mean'),
                sem_peak_dff0_global_min  = ('mean_peak_dff0_global_min', 'sem'),
                mean_peak_dff0_percentile = ('mean_peak_dff0_percentile', 'mean'),
                sem_peak_dff0_percentile  = ('mean_peak_dff0_percentile', 'sem'),
                mean_raw_intensity        = ('mean_raw_intensity',        'mean'),
                n_replicates              = ('replicate',                 'count'),
            )
            .reset_index())
    add_m('Condition_Summary', cond, '4A235B')

    add_m('README', pd.DataFrame({'Note': [
        'All_Peaks_Per_Cell    : every ROI from every video',
        'Technical_Replicates : mean per video (all cells averaged)',
        'Biological_Replicates: mean per replicate (N1/N2/N3), videos averaged first',
        'Condition_Summary     : mean±SEM across biological replicates',
        '',
        'Averaging order: cell → video mean → replicate mean → condition mean',
        'dff0_global_min : baseline = minimum intensity across full video',
        'dff0_percentile : baseline = 8th percentile across full video',
    ]}), '555555')

    master_excel = RESULTS_PATH / 'master_summary_simple.xlsx'
    wb.save(str(master_excel))
    print(f'\nMaster Excel saved: {master_excel}')

    # ── Bar plots ────────────────────────────────────────────────
    conditions = sorted(cond['condition'].unique())
    metrics = [
        ('mean_peak_dff0_global_min', 'sem_peak_dff0_global_min',
         'Peak dF/F₀ (global min)'),
        ('mean_peak_dff0_percentile', 'sem_peak_dff0_percentile',
         'Peak dF/F₀ (8th pct)'),
        ('mean_raw_intensity', None,
         'Mean raw intensity'),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    colors = plt.cm.Set2(np.linspace(0, 1, len(conditions)))
    for ax, (mean_col, sem_col, ylabel) in zip(axes, metrics):
        means = [cond[cond['condition']==c][mean_col].values[0]
                 for c in conditions]
        sems  = [cond[cond['condition']==c][sem_col].values[0]
                 for c in conditions] if sem_col else [0]*len(conditions)
        ax.bar(conditions, means, yerr=sems, capsize=5,
               color=colors[:len(conditions)], alpha=0.85)
        ax.set_ylabel(ylabel, fontsize=9)
        ax.set_title(ylabel, fontsize=9, fontweight='bold')
        ax.spines[['top', 'right']].set_visible(False)
    plt.suptitle('Condition Summary — mean ± SEM across biological replicates',
                 fontsize=11, fontweight='bold')
    plt.tight_layout()
    plot_path = RESULTS_PATH / 'cross_replicate_summary_simple.png'
    plt.savefig(str(plot_path), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Plot saved: {plot_path}')
    print('\n✅  Done!')